In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy

class Node():
    def __init__(self, 
                 feature_i_star=None, 
                 th_star=None, 
                 left=None, 
                 right=None, 
                 label=None):
        
        # for decision node
        self.feature_i_star = feature_i_star
        self.th_star = th_star
        self.left = left
        self.right = right
        
        # y_hat for leaf nodes
        self.label = label

class DecisionTreeClassifier():
    def __init__(self, max_depth=14, min_samples_split=35, min_info_gain=1e-7):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_info_gain = min_info_gain

    def build_tree(self, X, Y, depth=0):

        if len(np.unique(Y)) == 1:
            return Node(label=Y[0])

        # NEW: stop if too deep or too few samples to split
        if depth >= self.max_depth or len(Y) < self.min_samples_split:
            return Node(label=self.calculate_leaf_label(Y))

        n, m = X.shape

        feature_i_star, th_star, best_gain = self.get_best_split(X, Y)

        # NEW: stop if no split found, or best gain too small
        if feature_i_star == -1 or best_gain < self.min_info_gain:
             return Node(label=self.calculate_leaf_label(Y))

        feature_values = X[:, feature_i_star]
        left_mask = feature_values<=th_star
        right_mask = feature_values>th_star
        Yi_left = Y[left_mask]
        Yi_right = Y[right_mask]
        Xi_left = X[left_mask]
        Xi_right = X[right_mask]

        if len(Xi_left) == 0 or len(Xi_right) == 0:
            return Node(label=self.calculate_leaf_label(Y))

        left_subtree = self.build_tree(Xi_left, Yi_left, depth + 1)
        right_subtree = self.build_tree(Xi_right, Yi_right, depth + 1)
        
        return Node(feature_i_star, th_star, 
                    left_subtree, right_subtree)
    
    def get_best_split(self, X, Y):
        i_star, th_star = -1, -1
        max_info_gain = -float("inf")
        m = X.shape[-1]

        parent_entropy = self.entropy_calc(Y)

        for feature_i in range(m):
            feature_values = X[:, feature_i]
            thresholds = np.unique(feature_values)

            for th in thresholds:
                left_mask = feature_values<=th
                right_mask = feature_values>th
                Yi_left = Y[left_mask]
                Yi_right = Y[right_mask]
                Xi_left = X[left_mask]
                Xi_right = X[right_mask]

                if len(Xi_left)>0 and len(Xi_right)>0:
                    curr_info_gain = self.information_gain(parent_entropy, Yi_left, Yi_right)

                    if curr_info_gain>max_info_gain:
                        i_star = feature_i
                        th_star = th
                        max_info_gain = curr_info_gain

        return i_star, th_star, max_info_gain
    
    def entropy_calc(self, y):
         values, counts = np.unique(y, return_counts=True)
         p = counts / counts.sum()
         return entropy(p, base=2)

    def information_gain(self, parent_entropy, l_child, r_child):
       n_left, n_right = len(l_child), len(r_child)
       ita = n_left / (n_left + n_right)
       return parent_entropy - ita*self.entropy_calc(l_child) - (1-ita)*self.entropy_calc(r_child)
        
    def calculate_leaf_label(self, Y):
        values, counts = np.unique(Y, return_counts=True)
        return values[np.argmax(counts)]

    def fit(self, X, Y):
        self.root = self.build_tree(X, Y)

    def predict(self, X):
        predictions = []
        for x in X:
            y_hat  = self.make_prediction(x, self.root)
            predictions.append(y_hat)

        return predictions

    def make_prediction(self, x, tree):
        
        if tree.label is not None: #Leaf
            return tree.label
        
        feature_val = x[tree.feature_i_star]
        if feature_val<=tree.th_star:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)


In [20]:
# Carrega a base de dados a partir de seu caminho
print("Carregando dados...")
data = np.load("data/data.npz")
X_train = data["X_train"]
y_train = data["y_train"]
print("X_train carregado de forma: ", X_train.shape)
print("y_train carregado de forma: ", y_train.shape)

X_test = data['X_test']

classifier = DecisionTreeClassifier()
classifier.fit(X_train, y_train)
y_test = classifier.predict(X_test)

Carregando dados...
X_train carregado de forma:  (2831, 34)
y_train carregado de forma:  (2831,)


In [21]:
num_samples = X_test.shape[0]
submission_df = pd.DataFrame({
    'ID': np.arange(1, num_samples + 1),
    'Prediction': y_test
})

submission_df.to_csv("submission.csv", index=False)
print("Arquivo de submissão salvo em submission.csv")

Arquivo de submissão salvo em submission.csv


In [25]:
import numpy as np

# ============================================================
# ETAPA DE TUNING — NÃO FAZ PARTE DO NOTEBOOK/CÓDIGO FINAL
# Objetivo: escolher max_depth e min_samples_split usando
# apenas o X_train/y_train, sem tocar no X_test.
# ============================================================

def train_val_split(X, Y, val_ratio=0.2, seed=42):
    """Separa X_train/y_train em treino e validação."""
    rng = np.random.RandomState(seed)
    n = X.shape[0]
    indices = rng.permutation(n)

    n_val = int(n * val_ratio)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    X_tr, Y_tr = X[train_idx], Y[train_idx]
    X_val, Y_val = X[val_idx], Y[val_idx]
    return X_tr, Y_tr, X_val, Y_val


def accuracy(y_true, y_pred):
    y_pred = np.array(y_pred)
    return np.mean(y_true == y_pred)


# --- Carrega os dados (mesmo do notebook original) ---
data = np.load("data/data.npz")
X_train = data["X_train"]
y_train = data["y_train"]

# --- Separa treino/validação SÓ para o tuning ---
X_tr, y_tr, X_val, y_val = train_val_split(X_train, y_train, val_ratio=0.2, seed=42)

print(f"Treino (tuning): {X_tr.shape[0]} amostras")
print(f"Validação: {X_val.shape[0]} amostras")

# --- Combinações de hyperparâmetros a testar ---
# ajustado pro seu tamanho de dataset: 2831 amostras, 34 atributos
param_grid = [
    {"max_depth": 14,  "min_samples_split": 35},
    {"max_depth": 14,  "min_samples_split": 30},
    {"max_depth": 18, "min_samples_split": 35},
    {"max_depth": 14, "min_samples_split": 45},
    {"max_depth": 12, "min_samples_split": 30},
]

results = []
for params in param_grid:
    clf = DecisionTreeClassifier(
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_info_gain=1e-7,
    )
    clf.fit(X_tr, y_tr)

    y_val_pred = clf.predict(X_val)
    acc = accuracy(y_val, y_val_pred)

    results.append({**params, "val_accuracy": acc})
    print(f"max_depth={params['max_depth']:>3} | "
          f"min_samples_split={params['min_samples_split']:>3} | "
          f"acurácia validação = {acc:.4f}")

# --- Escolhe a melhor combinação ---
best = max(results, key=lambda r: r["val_accuracy"])
print("\nMelhor combinação encontrada:")
print(best)

Treino (tuning): 2265 amostras
Validação: 566 amostras
max_depth= 14 | min_samples_split= 35 | acurácia validação = 0.7527
max_depth= 14 | min_samples_split= 30 | acurácia validação = 0.7509
max_depth= 18 | min_samples_split= 35 | acurácia validação = 0.7527
max_depth= 14 | min_samples_split= 45 | acurácia validação = 0.7438
max_depth= 12 | min_samples_split= 30 | acurácia validação = 0.7509

Melhor combinação encontrada:
{'max_depth': 14, 'min_samples_split': 35, 'val_accuracy': np.float64(0.7526501766784452)}
